# FABQ-RC: Fisher-Adaptive Binary Quantization with Residual Codebooks

**Method spec:** see `FABQ_RC_SPEC.md` in this folder

This notebook:
1. Loads TinyLlama 1.1B as FP16 baseline
2. Implements FABQ-RC quantization
3. Converts to Q1_0_g128 for comparison
4. Evaluates perplexity on WikiText-2
5. Benchmarks downstream tasks

**Expected runtime:** ~30-45 min on Kaggle 2xT4

## 0. Setup & Imports

In [ ]:
!pip install -q --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q --force-reinstall transformers accelerate bitsandbytes scikit-learn
!pip install -q pandas numpy tqdm
!pip install -q llama-cpp-python  # CPU fallback for Q1_0_g128

import os
import math
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.cluster import MiniBatchKMeans
from tqdm.auto import tqdm
import pandas as pd

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pywavelets 1.5.0 requires numpy<2.0,>=1.22.4, but you have numpy 2.4.3 which is incompatible.
scipy 1.11.2 requires numpy<1.28.0,>=1.21.6, but you have numpy 2.4.3 which is incompatible.
cupy-cuda12x 12.2.0 requires numpy<1.27,>=1.20, but you have numpy 2.4.3 which is incompatible.
sentence-transformers 2.2.2 requires transformers<5.0.0,>=4.6.0, but you have transformers 5.7.0.dev0 which is incompatible.
gradient 2.0.6 requires attrs<=19, but you have attrs 23.1.0 which is incompatible.
matplotlib 3.7.3 requires numpy<2,>=1.20, but you have numpy 2.4.3 which is incompatible.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.4.3 which is incompatible.
pandas 2.2.0 requires numpy<2,>=1.23.2; python_version == "3.11", but you have numpy 2.4.3 which is incompatible.
tensorflow 2.15.0 requires numpy<2.0.0

## 1. Load Model & Tokenizer

In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

print("Loading FP16 model (baseline)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model.eval()

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params/1e9:.2f}B")
print(f"Trainable: {trainable/1e6:.1f}M")

Loading tokenizer...
Loading FP16 model (baseline)...


ImportError: 
AutoModelForCausalLM requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.


## 2. Prepare Calibration Dataset

In [ ]:
from datasets import load_dataset

print("Loading Pile subset for calibration...")
# Use a small subset of the Pile for calibration
pile_dataset = load_dataset("mit-han-lab/pile", split="train[:1%]")  # ~10MB for quick calibration

def tokenize_pile(batch):
    enc = tokenizer(
        batch['text'],
        truncation=True,
        max_length=512,
        padding='max_length'
    )
    enc['labels'] = enc['input_ids'].copy()
    return enc

print("Tokenizing calibration data...")
cal_dataset = pile_dataset.map(
    tokenize_pile,
    batched=True,
    remove_columns=['text']
)
cal_dataset.set_format('torch', columns=['input_ids', 'labels'])

cal_loader = DataLoader(cal_dataset, batch_size=4, shuffle=False)
print(f"Calibration batches: {len(cal_loader)}")

## 3. FABQ-RC Implementation

In [ ]:
class FisherAccumulator:
    """
    Accumulate Fisher information per output channel for each Linear layer.
    Uses gradient² as a proxy for Fisher Information.
    
    FIX: Use cross-entropy loss (not logits.mean()), and accumulate
    per-channel Fisher as sum of squared gradients (not square of sum).
    """
    def __init__(self, model):
        self.model = model
        self.fisher_accum = {}
        self.hooks = []
        self._register_hooks()

    def _register_hooks(self):
        def fisher_hook(module, grad_input, grad_output, name):
            if isinstance(module, (nn.Linear, nn.Conv2d)):
                if module.weight.grad is None:
                    return
                g = module.weight.grad.data
                if g.dim() == 2:
                    # Per-output-channel Fisher = sum of squared gradients over input dim
                    # (out_channels, in_channels) -> (out_channels,)
                    channel_fisher = (g ** 2).sum(dim=1)
                    if name not in self.fisher_accum:
                        self.fisher_accum[name] = torch.zeros_like(channel_fisher)
                    self.fisher_accum[name] += channel_fisher

        for name, module in self.model.named_modules():
            if isinstance(module, nn.Linear):
                h = module.register_full_backward_hook(
                    lambda m, gi, go: fisher_hook(m, gi, go, name)
                )
                self.hooks.append(h)

    def compute_fisher(self, dataloader, max_batches=50):
        """Run forward+backward on calibration data and compute Fisher per channel."""
        self.model.eval()
        for name in self.fisher_accum:
            self.fisher_accum[name].zero_()

        batch_count = 0
        for batch in tqdm(dataloader, desc="Computing Fisher", total=max_batches):
            if batch_count >= max_batches:
                break

            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)

            self.model.zero_grad()
            outputs = self.model(input_ids, labels=labels)
            # FIX: Use actual cross-entropy loss, not logits.mean()
            loss = outputs.loss
            loss.backward()

            batch_count += 1

        self.model.zero_grad()

        # Normalize
        for name in self.fisher_accum:
            self.fisher_accum[name] /= max(batch_count, 1)

        # Per-channel Fisher: already summed correctly in hook
        fisher_per_channel = {}
        for name, grad_sq in self.fisher_accum.items():
            fisher_per_channel[name] = grad_sq.sqrt()  # sqrt to convert squared back to linear scale

        return fisher_per_channel

    def remove_hooks(self):
        for h in self.hooks:
            h.remove()
        self.hooks = []


print("FisherAccumulator defined (FIXED).")

In [ ]:
class FABQLayer:
    """
    FABQ-RC quantized representation for a single Linear layer.
    """
    def __init__(self, name, weight, fisher_channels, config):
        self.name = name
        self.config = config
        self.out_channels, self.in_channels = weight.shape
        self.fisher_channels = fisher_channels  # (out_channels,)
        self.weight_fp32 = weight.clone()
        
        # Quantization outputs
        self.int8_channels = None
        self.int8_weights = None
        self.int8_scales = None
        self.binary_weights = None
        self.binary_scales = None
        self.blocksize = None
        self.codebook = None
        self.codebook_indices = None
        self.allocation = None
        
        self._quantize(weight)

    def _allocate_precision(self, sorted_channels, int8_fraction):
        """Stage 2: Assign int8 or binary per channel."""
        n_int8 = max(1, int(self.out_channels * int8_fraction))
        allocation = np.zeros(self.out_channels, dtype=np.uint8)
        int8_indices = []
        
        for rank, (channel_idx, _) in enumerate(sorted_channels):
            if rank < n_int8:
                allocation[channel_idx] = 1  # 1 = int8
                int8_indices.append(channel_idx)
        
        self.int8_channels = sorted(int8_indices)
        self.allocation = allocation

    def _select_blocksize(self, weights):
        """Stage 3: Pick best blocksize for binary channels."""
        candidates = self.config['blocksize_candidates']
        best_b, best_err = candidates[0], float('inf')
        
        for b in candidates:
            errors = []
            for start in range(0, self.in_channels, b):
                end = min(start + b, self.in_channels)
                block = weights[:, start:end]  # (nb, block_size)
                scale = block.std() + 1e-8
                block_q = np.where(block > 0, 1.0, -1.0)
                recon = block_q * scale
                err = ((block - recon) ** 2).mean()
                errors.append(err)
            total_err = np.mean(errors)
            if total_err < best_err:
                best_err = total_err
                best_b = b
        
        return best_b

    def _build_codebook(self, residual_blocks, n_clusters=256):
        """Stage 4: k-means on residual blocks."""
        if len(residual_blocks) == 0:
            return None, None
        
        # Flatten each block to a 1D vector
        flat = np.array([b.flatten() for b in residual_blocks], dtype=np.float32)
        
        kmeans = MiniBatchKMeans(
            n_clusters=min(n_clusters, len(residual_blocks)),
            random_state=42,
            batch_size=1024
        )
        labels = kmeans.fit_predict(flat)
        centroids = kmeans.cluster_centers_
        
        return centroids.astype(np.float32), labels

    def _quantize(self, weight):
        """Run all four FABQ-RC stages on a weight matrix."""
        cfg = self.config
        W = weight.cpu().numpy() if isinstance(weight, torch.Tensor) else weight
        
        # Sort channels by Fisher importance (descending)
        sorted_channels = sorted(
            enumerate(self.fisher_channels.cpu().numpy()),
            key=lambda x: x[1],
            reverse=True
        )
        
        # Stage 2: allocate precision
        self._allocate_precision(sorted_channels, cfg['int8_fraction'])
        
        # Extract int8 and binary channel subsets
        int8_mask = np.array([1 if c in set(self.int8_channels) else 0 
                              for c in range(self.out_channels)])
        int8_w = W[int8_mask == 1] if np.any(int8_mask) else None
        binary_w = W[int8_mask == 0] if np.any(int8_mask == 0) else None
        
        # Int8 quantization
        if int8_w is not None and len(int8_w) > 0:
            int8_scales = int8_w.std(axis=1, keepdims=True) + 1e-8
            self.int8_weights = np.round(int8_w / int8_scales).astype(np.int8)
            self.int8_scales = int8_scales.astype(np.float32)
        
        # Stage 3: select blocksize for binary channels
        if binary_w is not None and len(binary_w) > 0:
            binary_fisher = self.fisher_channels.cpu().numpy()[int8_mask == 0]
            self.blocksize = self._select_blocksize(binary_w)
            
            # Stage 4: residual codebook for binary channels
            residual_blocks = []
            binary_scales = []
            binary_q = np.zeros_like(binary_w)
            
            for start in range(0, self.in_channels, self.blocksize):
                end = min(start + self.blocksize, self.in_channels)
                block = binary_w[:, start:end]
                scale = block.std() + 1e-8
                block_q = np.where(block > 0, 1.0, -1.0)
                binary_q[:, start:end] = block_q
                binary_scales.append(scale)
                
                # Compute residual for codebook
                residual = block - block_q * scale
                residual_blocks.append(residual)
            
            self.binary_weights = binary_q.astype(np.float32)
            self.binary_scales = np.array(binary_scales, dtype=np.float32)
            
            self.codebook, self.codebook_indices = self._build_codebook(
                residual_blocks, 
                n_clusters=cfg['codebook_size']
            )
        else:
            self.blocksize = 128
            self.binary_weights = None
            self.binary_scales = None
            self.codebook = None
            self.codebook_indices = None

    def dequantize(self):
        """Reconstruct FP32 weight from quantized representation."""
        W = np.zeros((self.out_channels, self.in_channels), dtype=np.float32)
        
        # Reconstruct int8 channels
        if self.int8_weights is not None and len(self.int8_channels) > 0:
            for i, c in enumerate(self.int8_channels):
                W[c, :] = self.int8_weights[i].astype(np.float32) * self.int8_scales[i, 0]
        
        # Reconstruct binary channels
        if self.binary_weights is not None:
            binary_idx = 0
            for c in range(self.out_channels):
                if c not in set(self.int8_channels or []):
                    block_idx = 0
                    for start in range(0, self.in_channels, self.blocksize):
                        end = min(start + self.blocksize, self.in_channels)
                        block_q = self.binary_weights[binary_idx, start:end]
                        scale = self.binary_scales[block_idx]
                        
                        # Apply codebook correction if available
                        if self.codebook is not None and self.codebook_indices is not None:
                            correction = self.codebook[self.codebook_indices[block_idx]]
                            W[c, start:end] = block_q * scale + correction
                        else:
                            W[c, start:end] = block_q * scale
                        
                        block_idx += 1
                    binary_idx += 1
        
        return W
    
    def compression_ratio(self):
        """Compute effective bits per parameter."""
        int8_bits = 0
        binary_bits = 0
        scale_bits = 0
        codebook_bits = 0
        
        n_int8 = len(self.int8_channels or [])
        n_binary = self.out_channels - n_int8
        
        if n_int8 > 0:
            int8_bits = n_int8 * self.in_channels * 8
            scale_bits += n_int8 * 16  # FP16 scales
        
        if n_binary > 0:
            binary_bits = n_binary * self.in_channels * 1
            n_blocks = math.ceil(self.in_channels / self.blocksize)
            scale_bits += n_blocks * 16  # FP16 scales per block
            
            if self.codebook is not None:
                # Codebook: n_clusters * block_size * 32 bits, shared
                codebook_bits = self.codebook.size * 32
                codebook_idx_bits = n_blocks * math.log2(len(self.codebook))
                codebook_bits += codebook_idx_bits
        
        total_bits = int8_bits + binary_bits + scale_bits + codebook_bits
        total_params = self.out_channels * self.in_channels
        return total_bits / total_params


def quantize_model_fabq(model, fisher_per_channel, config):
    """
    Quantize all Linear layers in the model using FABQ-RC.
    
    Returns: dict of {layer_name: FABQLayer}
    """
    quantized_layers = {}
    
    for name, module in model.named_modules():
        if not isinstance(module, nn.Linear):
            continue
        
        if name not in fisher_per_channel:
            # Fallback: uniform Fisher
            fisher = torch.ones(module.out_features, device=fisher_per_channel[list(fisher_per_channel.keys())[0]].device)
        else:
            fisher = fisher_per_channel[name]
        
        print(f"  Quantizing {name}: {module.weight.shape}")
        
        ql = FABQLayer(name, module.weight.data.clone(), fisher, config)
        quantized_layers[name] = ql
    
    return quantized_layers


print("FABQLayer and quantize_model_fabq defined.")

## 4. Run FABQ-RC Quantization

In [ ]:
FABQ_CONFIG = {
    'int8_fraction': 0.05,          # 5% of channels at int8
    'blocksize_candidates': [16, 32, 64, 128, 256],
    'codebook_size': 256,           # residual codebook size
}

print("Step 1: Computing Fisher importance...")
accumulator = FisherAccumulator(model)
fisher_per_channel = accumulator.compute_fisher(cal_loader, max_batches=50)
accumulator.remove_hooks()

print(f"\nFisher computed for {len(fisher_per_channel)} layers")

# Show sample Fisher distribution
sample_name = list(fisher_per_channel.keys())[5]
f = fisher_per_channel[sample_name].cpu().numpy()
print(f"\nSample Fisher distribution ({sample_name}):")
print(f"  Mean: {f.mean():.6f}, Std: {f.std():.6f}")
print(f"  Min: {f.min():.6f}, Max: {f.max():.6f}")
print(f"  Top 10 channels: {sorted(f, reverse=True)[:10]}")

In [ ]:
print("\nStep 2: FABQ-RC quantization...")
quantized_layers = quantize_model_fabq(model, fisher_per_channel, FABQ_CONFIG)

# Report compression ratios
bpw_list = [l.compression_ratio() for l in quantized_layers.values()]
avg_bpw = np.mean(bpw_list)
print(f"\nFABQ-RC compression summary:")
print(f"  Average bpw: {avg_bpw:.3f}")
print(f"  Min bpw: {min(bpw_list):.3f}, Max bpw: {max(bpw_list):.3f}")
print(f"  Layers quantized: {len(quantized_layers)}")

## 5. Perplexity Evaluation

In [ ]:
from datasets import load_dataset

def compute_perplexity(model, tokenizer, dataset_name="wikitext",
                       dataset_config="wikitext-2-raw-v1",
                       split="test", max_seq_len=512, stride=256):
    """
    Compute perplexity on a text dataset.
    """
    print(f"Loading {dataset_name}/{dataset_config} {split}...")
    ds = load_dataset(dataset_name, dataset_config, split=split)
    
    enc = tokenizer("\n\n".join(ds['text']), return_tensors='pt', 
                    truncation=True, max_length=max_seq_len*2)
    input_ids = enc['input_ids']
    
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    
    for i in tqdm(range(0, len(input_ids) - 1, stride), desc="Evaluating perplexity"):
        input_chunk = input_ids[i:i+max_seq_len].to(device)
        target_chunk = input_ids[i+1:i+1+max_seq_len].to(device)
        
        with torch.no_grad():
            outputs = model(input_chunk, labels=target_chunk)
            loss = outputs.loss * target_chunk.numel()
            total_loss += loss.item()
            total_tokens += target_chunk.numel()
    
    perplexity = math.exp(total_loss / total_tokens)
    return perplexity, total_tokens


print("\n=== FP16 Baseline Perplexity ===")
# Reload FP16 model for clean baseline
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
)
fp16_ppl, fp16_tokens = compute_perplexity(model_fp16, tokenizer)
print(f"\nFP16 Perplexity: {fp16_ppl:.4f} ({fp16_tokens} tokens)")

In [ ]:
# Patch FABQ-RC weights into model and evaluate
print("\n=== FABQ-RC Perplexity ===")

# Create a copy of the FP16 model
model_fabq = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
)

# Replace weights with FABQ-RC reconstructed weights
for name, module in model_fabq.named_modules():
    if not isinstance(module, nn.Linear):
        continue
    if name in quantized_layers:
        ql = quantized_layers[name]
        W_recon = ql.dequantize()
        # Align to model's device
        module.weight.data = torch.from_numpy(W_recon).to(device=module.weight.device, dtype=torch.float16)

fabq_ppl, fabq_tokens = compute_perplexity(model_fabq, tokenizer)
print(f"\nFABQ-RC Perplexity: {fabq_ppl:.4f} ({fabq_tokens} tokens)")
print(f"Compression ratio: {avg_bpw:.3f} bpw")
print(f"Perplexity delta vs FP16: {fabq_ppl - fp16_ppl:.4f} ({(fabq_ppl/fp16_ppl - 1)*100:.2f}%)")

## 6. Downstream Task Evaluation

In [ ]:
from evaluate import load as load_metric

TASKS = ['hellaswag', 'arc_easy', 'arc_challenge', 'piqa']

def evaluate_tasks(model, tokenizer, tasks):
    """
    Evaluate model on multiple-choice benchmarks using lm-evaluation-harness style.
    For simplicity, we do few-shot prompting.
    """
    results = {}
    
    for task in tqdm(tasks, desc="Evaluating tasks"):
        try:
            metric = load_metric('accuracy')
            
            # Load task dataset
            if task == 'hellaswag':
                ds = load_dataset('hellaswag', split='val')
            elif task in ['arc_easy', 'arc_challenge']:
                ds = load_dataset(f'ai2_arc', f'{task}', split='validation')
            elif task == 'piqa':
                ds = load_dataset('piqa', split='validation')
            else:
                continue
            
            preds = []
            for example in tqdm(ds, desc=f'{task}', leave=False):
                prompt = _build_prompt(example, task)
                input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
                
                with torch.no_grad():
                    outputs = model.generate(
                        input_ids,
                        max_new_tokens=20,
                        temperature=0.0,
                        do_sample=False
                    )
                pred_text = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
                preds.append(_extract_answer(pred_text, example, task))
            
            refs = [_get_answer(example, task) for example in ds]
            accuracy = metric.compute(predictions=preds, references=refs)['accuracy']
            results[task] = accuracy
            
        except Exception as e:
            print(f"  Error on {task}: {e}")
            results[task] = None
    
    return results


def _build_prompt(example, task):
    if task == 'hellaswag':
        ctx = example['ctx']
        end = example['endings'][0]
        prompt = f"{ctx} {end}\n\nWhat happens next?"
    elif task in ['arc_easy', 'arc_challenge']:
        prompt = f"Question: {example['question']}\nChoices: {example['choices']['text']}\nAnswer:"
    elif task == 'piqa':
        prompt = f"Goal: {example['goal']}\nSolution:"
    return prompt

def _extract_answer(text, example, task):
    return text.strip().split('\n')[0][:50]  # truncate

def _get_answer(example, task):
    if task == 'hellaswag':
        return int(example['label'])
    elif task in ['arc_easy', 'arc_challenge']:
        return ord(example['answerKey']) - ord('A')
    elif task == 'piqa':
        return int(example['label'])


# NOTE: This cell may take a long time. Run selectively.
# Uncomment to run:
# print("\n=== FP16 Downstream Tasks ===")
# fp16_tasks = evaluate_tasks(model_fp16, tokenizer, TASKS)
# print("FP16:", fp16_tasks)
# print("\n=== FABQ-RC Downstream Tasks ===")
# fabq_tasks = evaluate_tasks(model_fabq, tokenizer, TASKS)
# print("FABQ-RC:", fabq_tasks)

## 7. Summary

In [ ]:
summary = pd.DataFrame([
    {
        'Method': 'FP16 (baseline)',
        'bpw': 16.0,
        'Perplexity': fp16_ppl,
        'Compression': '1x',
        'ARC-Easy': fp16_tasks.get('arc_easy', 'N/A') if 'fp16_tasks' in dir() else 'N/A',
        'HellaSwag': fp16_tasks.get('hellaswag', 'N/A') if 'fp16_tasks' in dir() else 'N/A',
    },
    {
        'Method': 'FABQ-RC (ours)',
        'bpw': round(avg_bpw, 3),
        'Perplexity': fabq_ppl,
        'Compression': f'{16.0/avg_bpw:.1f}x',
        'ARC-Easy': fabq_tasks.get('arc_easy', 'N/A') if 'fabq_tasks' in dir() else 'N/A',
        'HellaSwag': fabq_tasks.get('hellaswag', 'N/A') if 'fabq_tasks' in dir() else 'N/A',
    },
    {
        'Method': 'Q1_0_g128 (Bonsai)',
        'bpw': 1.125,
        'Perplexity': 'TODO',
        'Compression': '14.2x',
        'ARC-Easy': 'TODO',
        'HellaSwag': 'TODO',
    },
])

pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 120)
display(summary)

print("\n" + "="*60)
print("KEY INSIGHT: FABQ-RC uses adaptive per-layer blocksize + ")
print("residual codebook to preserve quality at ~1.15-1.20 bpw.")
print("="*60)